# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [ ]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

In [ ]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

## Helpers (use or modify)

In [ ]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [ ]:
SYMBOL = 'SPY'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage answers 200 OK with a prose blob when the free daily cap (25 calls)
    # is hit, so check for the series rather than trusting the status code.
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index':'date','4. close':'close'})[['date','close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['close'] = pd.to_numeric(df_api['close'])

if not USE_ALPHA:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False,
                         multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'], errors='coerce')

    print(df_api.head())
    print(df_api.dtypes)
    print("Shape:", df_api.shape)
    print("Missing values:")
    print(df_api.isna().sum())

v_api = validate(df_api, ['date','close']); v_api

In [ ]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [ ]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {'User-Agent': 'Mozilla/5.0'}

resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')

table = soup.find('table', id='constituents')

if table is None:
    raise ValueError("S&P 500 table not found")

header = [
    th.get_text(" ", strip=True)
    for th in table.find('tr').find_all('th')
]

data = []

for tr in table.find_all('tr')[1:]:
    cells = [
        td.get_text(" ", strip=True)
        for td in tr.find_all(['td', 'th'])
    ]

    if len(cells) == len(header):
        data.append(cells)

df_scrape = pd.DataFrame(data, columns=header)

df_scrape.head()


In [ ]:
df_scrape['CIK'] = pd.to_numeric(
    df_scrape['CIK'],
    errors='coerce'
)

df_scrape['Date added'] = pd.to_datetime(
    df_scrape['Date added'],
    errors='coerce'
)

In [ ]:
required = ['Symbol', 'Security', 'GICS Sector', 'CIK']

v_scrape = validate(df_scrape, required)

print(v_scrape)
print(df_scrape[required].dtypes)
print(df_scrape[required].isna().sum())
print("Duplicate symbols:", df_scrape['Symbol'].duplicated().sum())

In [ ]:
_ = save_csv(
    df_scrape,
    prefix='scrape',
    site='wikipedia',
    table='sp500_constituents'
)

## Documentation

### API Source
- Source: yfinance
- Ticker: SPY
- Period: 3 months
- Interval: Daily
- Data used: Date and Close

### Scrape Source
- Source: Wikipedia - List of S&P 500 companies
- Table: Current S&P 500 constituents

### Validation
For the API data, I checked that the date and close columns exist, converted them to the correct data types, and checked the shape and missing values.

For the scraped data, I checked required columns, missing values, data types, and duplicate ticker symbols.

### Assumptions & Risks
- yfinance availability or column names may change.
- The Wikipedia table structure may change in the future.
- S&P 500 constituents change over time.
- The data collected represents what was available when the notebook was run.

### Secrets
The `.env` file is stored locally and is not committed to GitHub.